<a href="https://colab.research.google.com/github/M1NG0LL/Driver-Drowsiness-Detection-DDD/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DataSet

installing dataset

In [2]:
import kagglehub
data_dir = kagglehub.dataset_download('ismailnasri20/driver-drowsiness-dataset-ddd')

print('Dataset downloaded to:', data_dir)

Using Colab cache for faster access to the 'driver-drowsiness-dataset-ddd' dataset.
Dataset downloaded to: /kaggle/input/driver-drowsiness-dataset-ddd


Split Data

In [3]:
!pip install split-folders

In [4]:
import splitfolders

data_dir = '/kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)'
output_dir = '/dataset/working'
splitfolders.ratio(data_dir, output=output_dir, seed=1337, ratio=(.8, 0.15, 0.05))

Copying files: 41793 files [05:55, 117.43 files/s]


In [5]:
train_dir = "/dataset/working/train"
test_dir = "/dataset/working/test"
val_dir = "/dataset/working/val"

# Importing

In [75]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, Input, GlobalAveragePooling2D,
    BatchNormalization, Activation, Concatenate, Add, MaxPooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, f1_score
)
from collections import Counter
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Data Augmentation

Variables

In [86]:
img_size = (300,300)
batch_size = 32

In [87]:
# train_datagen = ImageDataGenerator(
#     rescale=1./255,
#     rotation_range=30,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     shear_range=0.15,
#     zoom_range=0.3,
#     horizontal_flip=False,
#     brightness_range=[0.8, 1.2],
#     fill_mode='nearest'
# )

# test_val_datagen = ImageDataGenerator(rescale=1./255)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.9,1.1]
)
test_val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [88]:
train_batches = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
val_batches = test_val_datagen.flow_from_directory(
    val_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=True, seed=SEED
)
test_batches = test_val_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', shuffle=False, seed=SEED
)

Found 33434 images belonging to 2 classes.
Found 6268 images belonging to 2 classes.
Found 2091 images belonging to 2 classes.


In [89]:
print("\n--- Class Distribution ---")
for name, batches in zip(['Train', 'Validation', 'Test'],
                         [train_batches, val_batches, test_batches]):
    counts = Counter(batches.classes)
    print(f"{name}: {dict(counts)}")


--- Class Distribution ---
Train: {np.int32(0): 17878, np.int32(1): 15556}
Validation: {np.int32(0): 3352, np.int32(1): 2916}
Test: {np.int32(0): 1118, np.int32(1): 973}


# **Neural Network *(NN)***

Helper Methods

In [90]:
def conv_fn(x, filters, kernel, strides=1, padding='same'):
    """Conv2D + BatchNormalization + ReLU, returns the direct output."""
    x = Conv2D(filters, kernel, strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

Inception Customized Methods

In [91]:
def reduction_A_maker_fn(x):
    branch1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    branch2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(x)
    branch2 = BatchNormalization()(branch2)
    branch2 = Activation('relu')(branch2)

    branch3 = conv_fn(x, 256, 1, padding='same')
    branch3 = conv_fn(branch3, 256, 3, padding='same')
    branch3 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(branch3)
    branch3 = BatchNormalization()(branch3)
    branch3 = Activation('relu')(branch3)

    return Concatenate()([branch1, branch2, branch3])

def inception_resnet_B_maker_fn(x):
    in_channels = x.shape[-1]
    b1 = conv_fn(x, 192, 1)
    b2 = conv_fn(x, 128, 1)
    b2 = conv_fn(b2, 160, (1, 7))
    b2 = conv_fn(b2, 192, (7, 1))
    mixed = Concatenate()([b1, b2])
    mixed = Conv2D(in_channels, 1, padding='same', use_bias=True)(mixed)
    mixed = BatchNormalization()(mixed)
    return Activation('relu')(Add()([x, mixed]))

def reduction_B_maker_fn(x):
    b1 = MaxPooling2D((3, 3), strides=2, padding='valid')(x)

    b2 = conv_fn(x, 256, 1)
    b2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(b2)
    b2 = BatchNormalization()(b2)
    b2 = Activation('relu')(b2)

    b3 = conv_fn(x, 256, 1)
    b3 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b3)
    b3 = BatchNormalization()(b3)
    b3 = Activation('relu')(b3)

    b4 = conv_fn(x, 256, 1)
    b4 = conv_fn(b4, 256, 3, padding='same')
    b4 = Conv2D(256, 3, strides=2, padding='valid', use_bias=True)(b4)
    b4 = BatchNormalization()(b4)
    b4 = Activation('relu')(b4)

    return Concatenate()([b1, b2, b3, b4])


Build Hybrid Model

In [102]:
def build_model(input_shape=(300, 300, 3)):
    inputs = Input(shape=input_shape)
    backbone = EfficientNetB3(include_top=False, weights='imagenet', input_tensor=inputs)
    x = backbone.output

    # Add custom blocks
    x = reduction_A_maker_fn(x)
    x = inception_resnet_B_maker_fn(x)

    x = reduction_B_maker_fn(x)

    # Classification head
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model, backbone

In [103]:
model, backbone = build_model()
backbone.trainable = False

# **Train**

In [99]:
class_counts = Counter(train_batches.classes)
total = sum(class_counts.values())
class_weight = {
    0: total / (2 * class_counts[0]),
    1: total / (2 * class_counts[1])
}
print(f"\nClass weights: {class_weight}")


Class weights: {0: 0.935059850095089, 1: 1.07463358189766}


In [104]:
model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_auc', patience=6, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=4, mode='max', min_lr=1e-6),
    ModelCheckpoint('best_model_phase1.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

Phase 1: Training the Head

In [ ]:
history1 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks_phase1
)

Epoch 1/15
 188/1045 ━━━━━━━━━━━━━━━━━━━━ 12:28 874ms/step - accuracy: 0.8567 - auc: 0.9152 - loss: 0.4088 - precision: 0.8385 - recall: 0.8495

Phase 2: Fine-Tuning

In [ ]:
model = keras.models.load_model('best_model_phase1.h5', compile=False)

for layer in model.layers:
    layer.trainable = False

for layer in model.layers:
    if layer.name.startswith("block6") or layer.name.startswith("top") or layer.name.startswith("dense"):
        layer.trainable = True
        print(f"Layer {layer.name} is trainable")

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_auc', patience=8, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=5, mode='max', min_lr=1e-7),
    ModelCheckpoint('best_model_finetuned.h5', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

In [ ]:
history2 = model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=12,
    class_weight=class_weight,
    callbacks=callbacks_phase2
)

In [ ]:
model = keras.models.load_model('best_model_finetuned.h5')

# Plot Training Curves

In [ ]:
def plot_training(history, title):
    metrics = ['loss', 'accuracy', 'precision', 'recall', 'auc']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, metric in enumerate(metrics):
        ax = axes[i]
        ax.plot(history.history[metric], label='Train')
        ax.plot(history.history[f'val_{metric}'], label='Val')
        ax.set_title(f'{metric.upper()} - {title}')
        ax.legend()
    # Hide the sixth (empty) subplot
    axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

plot_training(history1, 'Phase 1')
plot_training(history2, 'Phase 2 (Fine-Tuning)')

# Evaluate

In [ ]:
print("\n--- Evaluation on Test Set ---")
test_loss, test_acc, test_prec, test_rec, test_auc = model.evaluate(test_batches, verbose=0)
print(f"Loss: {test_loss:.4f}, Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, AUC: {test_auc:.4f}")

y_pred_probs = model.predict(test_batches).flatten()
y_true = test_batches.classes


# Threshold Tuning

In [ ]:
val_probs = model.predict(val_batches).flatten()
val_true = val_batches.classes

precisions, recalls, thresholds = precision_recall_curve(val_true, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold based on F1-score (validation set): {best_thresh:.4f}")

y_pred_opt = (y_pred_probs > best_thresh).astype(int)

print("\n--- Classification Report (Default Threshold 0.5) ---")
print(classification_report(y_true, (y_pred_probs > 0.5).astype(int),
                            target_names=['Non Drowsy', 'Drowsy']))

print("\n--- Classification Report (Optimized Threshold) ---")
print(classification_report(y_true, y_pred_opt,
                            target_names=['Non Drowsy', 'Drowsy']))

# Plot confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_true, y_pred_opt), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non Drowsy', 'Drowsy'],
            yticklabels=['Non Drowsy', 'Drowsy'])
plt.title('Confusion Matrix (Optimized Threshold)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Test-Time Augmentation (TTA)

In [ ]:
def predict_with_tta(model, directory, target_size, batch_size, n_aug=5):
    """
    Predict using an average of n_aug random augmentations on the test data.
    Note: The directory must contain the same subfolder structure as the training data.
    """
    tta_datagen = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        rotation_range=10,
        zoom_range=0.1,
        brightness_range=[0.9, 1.1]
    )
    all_probs = []
    for i in range(n_aug):
        print(f"TTA round {i+1}/{n_aug}")
        gen = tta_datagen.flow_from_directory(
            directory,
            target_size=target_size,
            batch_size=batch_size,
            class_mode=None,      # we don't need labels
            shuffle=False
        )
        probs = model.predict(gen)
        all_probs.append(probs.flatten())
    return np.mean(all_probs, axis=0)


In [ ]:
print("\n--- Applying Test-Time Augmentation ---")
y_pred_tta_probs = predict_with_tta(model, test_dir, img_size, batch_size, n_aug=5)
y_pred_tta = (y_pred_tta_probs > best_thresh).astype(int)
print(classification_report(y_true, y_pred_tta))